In [10]:
import sys
!{sys.executable} -m pip install requests beautifulsoup4 pyyaml pandas -q

In [11]:
# ## Step 1: Load Config

In [12]:
import logging
import sys

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(name)s: %(message)s", stream=sys.stdout)

import yaml
from src.scraper import scrape_ipos
from src.rules import apply_filters
from src.models import AlertMessage
from src.notifier import send_telegram

In [ ]:
with open("config.yaml") as f:
    config = yaml.safe_load(f)

# Override: skip the "only last day" filter so we see all IPOs
# config["filters"]["only_last_day"] = False

print("Config loaded:")
for section, values in config.items():
    print(f"  {section}: {values}")

Config loaded:
  scraper: {'base_url': 'https://www.chittorgarh.com', 'year': 2026}
  filters: {'ipo_type': 'mainboard', 'only_last_day': False}
  alert_rules: {'min_total_subscription': 1.0, 'min_qib_subscription': 0, 'min_retail_subscription': 0}
  telegram: {'bot_token': '8718764683:AAHQnxqmxg5ojNxcxa0dfASWrgjZH2dx2H4', 'chat_id': 'YOUR_CHAT_ID_HERE'}


## Step 2: Scrape IPO subscription data from chittorgarh.com

In [14]:
all_ipos = scrape_ipos(config)
print(f"\nScraped {len(all_ipos)} mainboard IPOs")

2026-04-14 12:44:00,101 [INFO] src.scraper: Found 20 mainboard IPOs on dashboard
2026-04-14 12:44:00,102 [INFO] src.scraper: Fetching subscription details: https://www.chittorgarh.com/ipo_subscription/citius-transnet-invit-ipo/2862/
2026-04-14 12:44:00,217 [INFO] src.scraper: Fetching subscription details: https://www.chittorgarh.com/ipo_subscription/propshare-celestia-scheme-ipo/2965/
2026-04-14 12:44:00,320 [INFO] src.scraper: Fetching subscription details: https://www.chittorgarh.com/ipo_subscription/om-power-transmission-ipo/2676/
2026-04-14 12:44:00,445 [INFO] src.scraper: Fetching subscription details: https://www.chittorgarh.com/ipo_subscription/sai-parenterals-ipo/2681/
2026-04-14 12:44:00,568 [INFO] src.scraper: Fetching subscription details: https://www.chittorgarh.com/ipo_subscription/powerica-ipo/2570/
2026-04-14 12:44:00,701 [INFO] src.scraper: Fetching subscription details: https://www.chittorgarh.com/ipo_subscription/amir-chand-jagdish-kumar-ipo/2501/
2026-04-14 12:44:00

## Step 3: View all IPOs as a table

In [15]:
import pandas as pd

rows = []
for ipo in all_ipos:
    rows.append({
        "IPO Name": ipo.name,
        "Close Date": ipo.close_date.strftime("%d %b %Y"),
        "QIB (x)": ipo.qib_subscription,
        "sNII (x)": ipo.snii_subscription,
        "bNII (x)": ipo.bnii_subscription,
        "Retail (x)": ipo.retail_subscription,
        "Total (x)": ipo.total_subscription,
    })

df = pd.DataFrame(rows).sort_values("Total (x)", ascending=False).reset_index(drop=True)
df.index += 1
df

,IPO Name,Close Date,QIB (x),sNII (x),bNII (x),Retail (x),Total (x)
1,Shree Ram Twistex,25 Feb 2026,3.9424,72.84,41.29,76.6320,43.664300
2,Gaudium IVF & Women Health,24 Feb 2026,1.6244,5.23,0.00,7.5980,5.091900
3,Raajmarg Infra Investment Trust,13 Mar 2026,19.3374,0.00,0.00,0.0000,4.885800
4,Innovision,17 Mar 2026,14.3037,0.59,3.35,0.6023,3.457900
5,Om Power Transmission,13 Apr 2026,3.6500,0.00,0.00,1.5400,3.330000
6,Amir Chand Jagdish Kumar (Exports),27 Mar 2026,1.1751,3.05,0.00,1.4389,2.941000
7,SEDEMAC Mechatronics,06 Mar 2026,8.4587,2.19,0.00,0.2010,1.879200
8,Shadowfax Technologies,22 Jan 2026,3.9988,2.02,0.00,2.4313,1.576700
9,Fractal Analytics,11 Feb 2026,4.3861,2.01,0.00,1.0985,1.570000
10,GSP Crop Science,18 Mar 2026,2.6585,1.44,0.00,0.4173,1.148900


## Step 4: Apply filters (configurable thresholds)

In [16]:
matched = apply_filters(all_ipos, config)
print(f"{len(matched)} IPO(s) passed the alert rules\n")

for ipo in matched:
    print(f"  {ipo.name:40s}  Total: {ipo.total_subscription:.2f}x")

2026-04-14 12:44:02,649 [INFO] src.rules: Mainboard filter: 20 -> 20 IPOs
2026-04-14 12:44:02,650 [INFO] src.rules: Total subscription >= 1.0x: 20 -> 12 IPOs
12 IPO(s) passed the alert rules

  Om Power Transmission                     Total: 3.33x
  Powerica                                  Total: 1.07x
  Amir Chand Jagdish Kumar (Exports)        Total: 2.94x
  GSP Crop Science                          Total: 1.15x
  Raajmarg Infra Investment Trust           Total: 4.89x
  Innovision                                Total: 3.46x
  Rajputana Stainless                       Total: 1.12x
  SEDEMAC Mechatronics                      Total: 1.88x
  Shree Ram Twistex                         Total: 43.66x
  Gaudium IVF & Women Health                Total: 5.09x
  Fractal Analytics                         Total: 1.57x
  Shadowfax Technologies                    Total: 1.58x


## Step 5: Preview the alert message

In [17]:
alert = AlertMessage(ipos=matched)
print(alert.format())

IPO Subscription Alert
──────────────────────────────
*Om Power Transmission*
Close Date: 13 Apr 2026
Type: Mainboard
QIB: 3.65x
sNII: 0.00x
bNII: 0.00x
Retail: 1.54x
Total: 3.33x
──────────────────────────────
*Powerica*
Close Date: 27 Mar 2026
Type: Mainboard
QIB: 4.74x
sNII: 1.38x
bNII: 0.00x
Retail: 0.16x
Total: 1.07x
──────────────────────────────
*Amir Chand Jagdish Kumar (Exports)*
Close Date: 27 Mar 2026
Type: Mainboard
QIB: 1.18x
sNII: 3.05x
bNII: 0.00x
Retail: 1.44x
Total: 2.94x
──────────────────────────────
*GSP Crop Science*
Close Date: 18 Mar 2026
Type: Mainboard
QIB: 2.66x
sNII: 1.44x
bNII: 0.00x
Retail: 0.42x
Total: 1.15x
──────────────────────────────
*Raajmarg Infra Investment Trust*
Close Date: 13 Mar 2026
Type: Mainboard
QIB: 19.34x
sNII: 0.00x
bNII: 0.00x
Retail: 0.00x
Total: 4.89x
──────────────────────────────
*Innovision*
Close Date: 17 Mar 2026
Type: Mainboard
QIB: 14.30x
sNII: 0.59x
bNII: 3.35x
Retail: 0.60x
Total: 3.46x
──────────────────────────────
*Rajputa

## Step 6: Send via Telegram (uncomment to send for real)

In [18]:
# Uncomment the line below to actually send the alert to Telegram
# (make sure bot_token and chat_id are set in config.yaml first)

# send_telegram(alert, config)